# L6a Advanced: Single Index Model Estimation Theory
This deeper dive develops the mathematical chain behind the L6a and L6b examples beyond what the lecture states: regularized (ridge) estimation of the single index model and its sampling covariance, the two bootstraps and what each assumes, the unit conventions that must agree before a SIM covariance is assembled, the propagation of parameter uncertainty into portfolio risk and weights, and the maximum-Sharpe allocation as a second-order cone program. The notation follows the lecture: growth rates $g_{i,t}$ in inverse years, market index $M$ (SPY), $N$ observations, and $|\mathcal{P}|$ assets.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> - **Derive the ridge estimator and its covariance, with the caveats:** Write the regularized least-squares problem, its closed-form solution, and its model-based covariance, and state why the classical degrees of freedom and Student t intervals do not carry over unchanged to a regularized fit.
> - **Distinguish the two bootstraps and keep the units straight:** Explain what the empirical-residual and Gaussian parametric bootstraps preserve and discard, and convert consistently between the growth-rate, log-return, and diffusion-volatility conventions of a SIM covariance.
> - **Propagate parameter uncertainty and pose the maximum-Sharpe problem:** Explain how joint parameter draws induce distributions of portfolio risk and weights, and reformulate the long-only maximum-Sharpe problem as a second-order cone program.

Let's get started!
___

## 1. SIM Regression and Assumptions
For asset $i$ and the market index $M$ on trading day $t$, the single index model of the lecture reads:
$$
g_{i,t}=\alpha_{i}+\beta_{i}\,g_{M,t}+\varepsilon_{i,t}
$$
with $\alpha_{i}$ and $\varepsilon_{i,t}$ in growth-rate units (inverse years) and $\beta_{i}$ dimensionless. The SIM assumes $\mathbb{E}[\varepsilon_{i,t}]=0$, $\text{Cov}(g_{M,t},\varepsilon_{i,t})=0$, and $\text{Cov}(\varepsilon_{i,t},\varepsilon_{j,t})=0$ for $i\neq j$. These assumptions route contemporaneous cross-asset covariance through the market factor. They do not imply Gaussian tails, temporal independence, or constant future parameters; those are separate modeling choices that must be diagnosed. For the regression inference below we also condition on the market series and assume $\mathbb{E}[\varepsilon_{i,t}\,|\,g_{M,\cdot}]=0$.
___

## 2. One Model, Three Unit Conventions
The course notebooks observe annualized continuously compounded growth rates $g_{t}=\frac{1}{\Delta{t}}\ln(S_{t}/S_{t-1})$ (L3a, L5a). The fitted residual standard deviation is therefore a growth-rate standard deviation, $s_{g,\varepsilon}$, and the covariance of growth-rate observations contains $s^{2}_{g,\varepsilon}$ directly, with no time-step factor.

For one-step log returns $r_{t}=\Delta{t}\,g_{t}$ the covariance is $\mathbf{\Sigma}_{r}=\Delta{t}^{2}\,\mathbf{\Sigma}_{g}$, and if a diffusion is written as $\text{Var}(r_{t})=\Delta{t}\,\mathbf{C}$ (the GBM covariance rate of L5a) then $\mathbf{C}=\Delta{t}\,\mathbf{\Sigma}_{g}$ and the diffusion volatility is $\sqrt{\Delta{t}}\,\sigma_{g}$. All three are valid conventions. The error is to mix a market variance from one convention with an idiosyncratic variance from another, or to insert an isolated $\Delta{t}$ into only one term of the SIM covariance.
___

## 3. Regularized Estimation and Its Covariance
Let $\hat{\mathbf{X}}=[\mathbf{1}\;\mathbf{g}_{M}]\in\mathbb{R}^{N\times2}$, $\mathbf{y}_{i}=\mathbf{g}_{i}$, and $\boldsymbol{\theta}_{i}=(\alpha_{i},\beta_{i})^{\top}$. Ridge estimation adds a penalty on the parameters to the least-squares objective. Because the intercept should not be shrunk (a penalized intercept biases the fitted mean and the residuals no longer sum to zero), the penalty acts on the slope only, through $\mathbf{\Lambda}=\text{diag}(0,1)$:
$$
\hat{\boldsymbol{\theta}}_{i}=\arg\min_{\boldsymbol{\theta}}\left\{\tfrac{1}{2}\lVert\mathbf{y}_{i}-\hat{\mathbf{X}}\boldsymbol{\theta}\rVert_{2}^{2}+\tfrac{\delta}{2}\,\boldsymbol{\theta}^{\top}\mathbf{\Lambda}\boldsymbol{\theta}\right\}
\quad\Longrightarrow\quad
\boxed{\hat{\boldsymbol{\theta}}_{i}=(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\delta\mathbf{\Lambda})^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}_{i}}
$$
where $\delta\geq0$ is the ridge parameter; $\delta=0$ recovers ordinary least squares. A numeric $\delta$ has no scale-free meaning here, because the market column has growth-rate units while the intercept column is dimensionless: with an unpenalized intercept the slope information is the centered sum $\sum_{t}(g_{M,t}-g^{\prime}_{M})^{2}$, which sets the scale of $\delta$, and centering the market series before penalizing the slope is the usual practice. Substituting the model $\mathbf{y}_{i}=\hat{\mathbf{X}}\boldsymbol{\theta}_{i}+\boldsymbol{\varepsilon}$ shows the estimator is biased for $\delta>0$: with $\mathbf{K}^{-1}=(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}+\delta\mathbf{\Lambda})^{-1}$,
$$
\hat{\boldsymbol{\theta}}_{i}=\underbrace{\mathbf{K}^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}}_{\text{shrinkage matrix}}\,\boldsymbol{\theta}_{i}+\mathbf{K}^{-1}\hat{\mathbf{X}}^{\top}\boldsymbol{\varepsilon}
$$
so $\mathbb{E}[\hat{\boldsymbol{\theta}}_{i}]\neq\boldsymbol{\theta}_{i}$ while the variance of the second term is smaller than the least-squares variance: the bias-variance tradeoff. Under the homoskedastic model $\text{Cov}(\boldsymbol{\varepsilon})=\sigma^{2}_{g,\varepsilon}\mathbf{I}$, the covariance of the estimator is:
$$
\boxed{\text{Cov}(\hat{\boldsymbol{\theta}}_{i})=\sigma^{2}_{g,\varepsilon}\,\mathbf{K}^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\,\mathbf{K}^{-1}}
$$
Only for $\delta=0$ does this reduce to $\sigma^{2}_{g,\varepsilon}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}$; dropping the middle Gram matrix for nonzero ridge is an algebraic error. Two further caveats. This is the __model-based__ covariance under homoskedastic, uncorrelated errors; a heteroskedasticity- and autocorrelation-consistent covariance replaces the middle factor by $\hat{\mathbf{X}}^{\top}\hat{\mathbf{\Omega}}\hat{\mathbf{X}}$ with an estimate $\hat{\mathbf{\Omega}}$ of the error covariance, and the daily growth rates of this course (heavy tails, lag-one dependence, clustered variance) are a case for it. And the classical inference of the lecture does not carry over unchanged. With the ridge hat matrix $\mathbf{H}_{\delta}=\hat{\mathbf{X}}\mathbf{K}^{-1}\hat{\mathbf{X}}^{\top}$, the residual sum of squares no longer has $N-2$ degrees of freedom: its noise part has expectation $\sigma^{2}_{g,\varepsilon}\left(N-2\,\text{tr}(\mathbf{H}_{\delta})+\text{tr}(\mathbf{H}_{\delta}^{\top}\mathbf{H}_{\delta})\right)$ and a bias term adds to it, so dividing by $N-2$ (or by the effective model degrees of freedom $N-\text{tr}(\mathbf{H}_{\delta})$) is a heuristic; the estimator is biased; and an exact Student $t$ pivot around $\boldsymbol{\theta}_{i}$ does not exist. Intervals for a regularized fit are better obtained by the bootstraps of the next section. The course package's `estimate_sim` and `bootstrap_sim` implement the ridge with a penalty on both coefficients and $N-2$ in the residual variance; that is exact for the $\delta=0$ case the core examples use, and approximate otherwise.
___

## 4. Bootstrap Uncertainty Quantification
Starting from the fitted values $\hat{\mathbf{y}}=\hat{\mathbf{X}}\hat{\boldsymbol{\theta}}_{i}$, each bootstrap replicate constructs $\mathbf{y}^{(b)}=\hat{\mathbf{y}}+\boldsymbol{\varepsilon}^{(b)}$ and refits the model:

- __Empirical-residual bootstrap:__ sample the centered fitted residuals with replacement. This preserves the observed one-step marginal shape, including asymmetry and heavy tails.
- __Gaussian parametric bootstrap:__ draw $\boldsymbol{\varepsilon}^{(b)}\sim\mathcal{N}(\mathbf{0},s^{2}_{g,\varepsilon}\mathbf{I})$. This isolates uncertainty under the Gaussian innovation assumption.

The empirical quantiles of $\{\hat{\alpha}^{(b)},\hat{\beta}^{(b)},s^{(b)}_{g,\varepsilon}\}$ give percentile intervals, and comparing the two methods is a model-risk diagnostic. Both procedures hold the market series fixed, assume the fitted model is the right conditional mean, and draw innovations independently across days (exchangeable, homoskedastic); neither retains the volatility clustering diagnosed in L3a, nor the lag-one dependence that growth rates built from volume-weighted average prices carry, nor, when run security by security, the residual correlation across securities. Moving-block, stationary, or regime-aware bootstraps are required when multi-day persistence is part of the estimand. For the Gaussian procedure with $\delta=0$, the limiting covariance is exactly the classical $s^{2}_{g,\varepsilon}(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}$; the [executable L6a notebook](../../CHEME-5660-L6a-Example-SIM-Parameter-Uncertainty-Fall-2026.ipynb) compares empirical and theoretical standard errors.
___

## 5. SIM Covariance as Rank One Plus Diagonal
Under the SIM orthogonality assumptions the growth-rate covariance is (lecture):
$$
\boxed{\mathbf{\Sigma}_{g}^{\text{SIM}}=\sigma^{2}_{g,M}\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top}+\text{diag}\left(\sigma^{2}_{g,\varepsilon,1},\ldots,\sigma^{2}_{g,\varepsilon,|\mathcal{P}|}\right)}
$$
so that $(\Sigma_{g})_{ij}=\beta_{i}\beta_{j}\sigma^{2}_{g,M}$ for $i\neq j$ and $(\Sigma_{g})_{ii}=\beta_{i}^{2}\sigma^{2}_{g,M}+\sigma^{2}_{g,\varepsilon,i}$. A full covariance has $|\mathcal{P}|(|\mathcal{P}|+1)/2$ distinct entries; the SIM uses $|\mathcal{P}|$ betas, $|\mathcal{P}|$ idiosyncratic variances, and one market variance. For $|\mathcal{P}|=500$ that is $1{,}001$ rather than $125{,}250$ covariance parameters (the mean vector adds $|\mathcal{P}|$ intercepts and the market mean). The reduction stabilizes estimation but imposes a strong diagonal-residual assumption that must be checked; the estimation example measures the residual correlations in the course dataset and finds them small for most pairs, a few tenths for related firms, and near one for near-duplicate securities.
___

## 6. From Parameter Uncertainty to Decision Uncertainty
For each bootstrap scenario $b$, preserve the joint within-asset draw $(\alpha_{i}^{(b)},\beta_{i}^{(b)},s^{(b)}_{g,\varepsilon,i})$, construct $(\bar{\mathbf{g}}^{(b)},\mathbf{\Sigma}_{g}^{(b)})$ with the training-period market mean and variance, and solve for scenario weights $\mathbf{w}^{(b)}$. Compare them with the deployed point-estimate weights $\hat{\mathbf{w}}$. Useful diagnostics include:
$$
\text{fixed variance}^{(b)}=\hat{\mathbf{w}}^{\top}\mathbf{\Sigma}_{g}^{(b)}\hat{\mathbf{w}},\qquad
\text{variance regret}^{(b)}=\hat{\mathbf{w}}^{\top}\mathbf{\Sigma}_{g}^{(b)}\hat{\mathbf{w}}-(\mathbf{w}^{(b)})^{\top}\mathbf{\Sigma}_{g}^{(b)}\mathbf{w}^{(b)},\qquad
\text{allocation distance}^{(b)}=\tfrac{1}{2}\lVert\mathbf{w}^{(b)}-\hat{\mathbf{w}}\rVert_{1}
$$
This layer matters because optimization is nonlinear: individually narrow coefficient intervals can still generate unstable weights (L5b's advanced estimation-risk notebook made the same point for the full-covariance optimizer). The [L6b advanced propagation notebook](../../../L6b/advanced/uncertainty/CHEME-5660-L6b-Advanced-SIM-Portfolio-Uncertainty-Fall-2026.ipynb) implements this workflow for an unconstrained minimum-variance portfolio.
___

## 7. Maximum Sharpe Ratio as a Second-Order Cone Program
Let $\mathbf{c}=\bar{\mathbf{g}}-g_{f}\mathbf{1}$ be the vector of expected excess growth rates and factor the covariance as $\mathbf{\Sigma}_{g}=\mathbf{U}^{\top}\mathbf{U}$ (a Cholesky factor, L5a). The Sharpe ratio of the lecture L5b, in growth-rate units, is:
$$
\text{SR}(\mathbf{w})=\frac{\mathbf{c}^{\top}\mathbf{w}}{\lVert\mathbf{U}\mathbf{w}\rVert_{2}}
$$
Its maximization over long-only, fully invested weights is not a convex program as written (a ratio), but it becomes one after a change of variables. Assume $\mathbf{\Sigma}_{g}$ is positive definite (so no feasible portfolio has zero risk) and that some feasible portfolio has $\mathbf{c}^{\top}\mathbf{w}>0$. Set $\mathbf{y}=\tau\,\mathbf{w}$ with $\tau=1/(\mathbf{c}^{\top}\mathbf{w})>0$, so that $\mathbf{c}^{\top}\mathbf{y}=1$; then $\text{SR}(\mathbf{w})=1/\lVert\mathbf{U}\mathbf{y}\rVert_{2}$, and maximizing it is:
$$
\boxed{\min_{\mathbf{y},\tau}\;\lVert\mathbf{U}\mathbf{y}\rVert_{2}\quad\text{subject to}\quad\mathbf{c}^{\top}\mathbf{y}=1,\qquad\mathbf{1}^{\top}\mathbf{y}=\tau,\qquad\mathbf{y}\geq0}
$$
Positivity of $\tau$ is implied ($\mathbf{y}\geq0$ with $\mathbf{c}^{\top}\mathbf{y}=1$ forces $\mathbf{y}\neq\mathbf{0}$), so it needs no constraint of its own, and the recovered portfolio is $\mathbf{w}=\mathbf{y}/\tau$. The norm objective is a second-order cone; the remaining constraints are linear. The course package solves a related cone formulation (a minimum-Sharpe constraint $[\mathbf{c}^{\top}\mathbf{w}/\tau;\,\mathbf{U}\mathbf{w}]$ in a second-order cone, with the COSMO solver) in the L7b and L13a examples.
___

## Summary
This notebook developed the theory behind the L6a and L6b examples: the ridge estimator and its model-based covariance with the caveats that regularization brings, the two bootstraps and their shared independence assumption, the unit conventions of a SIM covariance, the propagation of parameter draws into weights, and the maximum-Sharpe problem as a cone program.

> __Key Takeaways:__
>
> - **Regularization changes the covariance and the inference:** The ridge estimator is biased with a smaller variance, its covariance keeps the middle Gram matrix, and the classical degrees of freedom and Student t intervals do not carry over unchanged, so bootstrap intervals are the practical route for a regularized fit.
> - **Bootstraps and units are assumptions:** Residual and parametric bootstraps differ in the innovation shape but share independent draws across days and securities, and a SIM covariance must be assembled in one convention (growth rate, log return, or covariance rate) throughout.
> - **Decision uncertainty is the endpoint:** Joint parameter draws propagated through covariance construction and optimization reveal weight instability that coefficient intervals hide, and the long-only maximum-Sharpe allocation that uses those inputs is a second-order cone program.

The L6b examples use these pieces to build and stress SIM-based portfolios.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___